In [ ]:
# Required code provided - copy exactly:

# Cardiovascular Disease Risk Prediction Dataset
import pandas as pd
import numpy as np
import altair as alt
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Configure Altair
alt.data_transformers.enable("vegafusion")
alt.renderers.enable('colab')

def generate_cardiovascular_dataset(n_samples=1000):
    """
    Generate synthetic cardiovascular disease risk dataset

    Features based on established cardiovascular risk factors:
    - Age, BMI, blood pressure, cholesterol levels
    - Smoking history, diabetes status, family history
    - Exercise frequency, stress levels
    """
    np.random.seed(42)

    # Realistic cardiovascular disease rate (~30% high risk)
    n_high_risk = int(n_samples * 0.30)
    n_low_risk = n_samples - n_high_risk

    # Age (high risk patients tend to be older)
    age_high = np.random.normal(loc=58, scale=12, size=n_high_risk)
    age_low = np.random.normal(loc=45, scale=15, size=n_low_risk)
    age = np.concatenate([age_high, age_low])

    # BMI (Body Mass Index)
    bmi_high = np.random.normal(loc=29.5, scale=5, size=n_high_risk)
    bmi_low = np.random.normal(loc=25.8, scale=4, size=n_low_risk)
    bmi = np.concatenate([bmi_high, bmi_low])

    # Systolic Blood Pressure (mmHg)
    bp_high = np.random.normal(loc=145, scale=20, size=n_high_risk)
    bp_low = np.random.normal(loc=125, scale=18, size=n_low_risk)
    systolic_bp = np.concatenate([bp_high, bp_low])

    # Total Cholesterol (mg/dL)
    chol_high = np.random.normal(loc=240, scale=35, size=n_high_risk)
    chol_low = np.random.normal(loc=195, scale=30, size=n_low_risk)
    cholesterol = np.concatenate([chol_high, chol_low])

    # Smoking (0=Never, 1=Former, 2=Current)
    smoking_high = np.random.choice([0, 1, 2], size=n_high_risk, p=[0.3, 0.4, 0.3])
    smoking_low = np.random.choice([0, 1, 2], size=n_low_risk, p=[0.6, 0.3, 0.1])
    smoking_status = np.concatenate([smoking_high, smoking_low])

    # Diabetes (0=No, 1=Yes)
    diabetes_high = np.random.choice([0, 1], size=n_high_risk, p=[0.6, 0.4])
    diabetes_low = np.random.choice([0, 1], size=n_low_risk, p=[0.9, 0.1])
    diabetes = np.concatenate([diabetes_high, diabetes_low])

    # Family History (0=No, 1=Yes)
    family_high = np.random.choice([0, 1], size=n_high_risk, p=[0.4, 0.6])
    family_low = np.random.choice([0, 1], size=n_low_risk, p=[0.7, 0.3])
    family_history = np.concatenate([family_high, family_low])

    # Exercise Hours per Week
    exercise_high = np.random.normal(loc=2.5, scale=2, size=n_high_risk)
    exercise_low = np.random.normal(loc=4.8, scale=2.5, size=n_low_risk)
    exercise_hours = np.concatenate([exercise_high, exercise_low])

    # Create target variable
    cardiovascular_risk = np.concatenate([
        np.ones(n_high_risk, dtype=int),    # High risk = 1
        np.zeros(n_low_risk, dtype=int)     # Low risk = 0
    ])

    # Combine into dataset
    dataset = pd.DataFrame({
        'age': np.clip(age, 18, 85).round(0),
        'bmi': np.clip(bmi, 15, 45).round(1),
        'systolic_bp': np.clip(systolic_bp, 90, 200).round(0),
        'cholesterol': np.clip(cholesterol, 120, 350).round(0),
        'smoking_status': smoking_status,
        'diabetes': diabetes,
        'family_history': family_history,
        'exercise_hours_per_week': np.clip(exercise_hours, 0, 15).round(1),
        'cardiovascular_risk': cardiovascular_risk
    })

    # Shuffle dataset
    dataset = dataset.sample(frac=1).reset_index(drop=True)

    return dataset

# Generate the dataset
cardio_data = generate_cardiovascular_dataset(1000)

This data is a synthetic cardiovascula disease risk dataset that contains 1000 samples, with approximately 30% classified as high-risk and 70% as low risk for cardiovascular disease. It includes 8 features which are Age, BMI, Systolic blood pressure, Cholesterol, Smoking status, Diabities, Family history, and Exercise hours per week. This data is generated using realistic distributions based on established risk factors.High-risk individuals tend to be older, have higher BMI, elevated blood pressure, and higher cholesterol, with a greater likelihood of smoking, diabetes, and family history of cardiovascular issues.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Defining feature columns and target
feature_columns = ['age', 'bmi', 'systolic_bp', 'cholesterol', 'smoking_status', 'diabetes', 'family_history', 'exercise_hours_per_week']
X = cardio_data[feature_columns]
y = cardio_data['cardiovascular_risk']

# Performing train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Printing shapes
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# Printing target distribution
print("\nTarget distribution in y_train:")
print(y_train.value_counts(normalize=True))
print("\nTarget distribution in y_test:")
print(y_test.value_counts(normalize=True))

X_train shape: (800, 8)
X_test shape: (200, 8)
y_train shape: (800,)
y_test shape: (200,)

Target distribution in y_train:
cardiovascular_risk
0    0.7
1    0.3
Name: proportion, dtype: float64

Target distribution in y_test:
cardiovascular_risk
0    0.7
1    0.3
Name: proportion, dtype: float64


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Defining algorithms with preprocessing pipelines
algorithms = {
    'KNN': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    'DecisionTree': make_pipeline(StandardScaler(), DecisionTreeClassifier(max_depth=8, random_state=42)),
    'RandomForest': make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=100, random_state=42))
}

In [ ]:
from sklearn.model_selection import cross_validate
import numpy as np

# Define scoring metrics
scoring = ['accuracy', 'precision', 'recall']

# Perform 5-fold cross-validation for each algorithm
comparison_results = {}
for name, model in algorithms.items():
    cv_results = cross_validate(model, X, y, cv=5, scoring=scoring, return_train_score=False)
    comparison_results[name] = {
        'accuracy': cv_results['test_accuracy'],
        'precision': cv_results['test_precision'],
        'recall': cv_results['test_recall']
    }

# Print mean ± std for each metric for each algorithm
for name, results in comparison_results.items():
    print(f"\nResults for {name}:")
    for metric in scoring:
        mean_score = np.mean(results[metric])
        std_score = np.std(results[metric])
        print(f"{metric.capitalize()}: {mean_score:.4f} ± {std_score:.4f}")


Results for KNN:
Accuracy: 0.8930 ± 0.0216
Precision: 0.8520 ± 0.0566
Recall: 0.7833 ± 0.0408

Results for DecisionTree:
Accuracy: 0.8580 ± 0.0319
Precision: 0.7745 ± 0.0523
Recall: 0.7433 ± 0.0720

Results for RandomForest:
Accuracy: 0.8950 ± 0.0179
Precision: 0.8760 ± 0.0515
Recall: 0.7600 ± 0.0170


In [ ]:
# 1. Exploratory Data Analysis (EDA) & Visualization
# Age Distribution by Cardiovascular Risk
age_chart = alt.Chart(cardio_data).mark_bar().encode(
    x=alt.X('age', bin=True, title='Age (years)'),
    y=alt.Y('count()', title='Number of Individuals'),
    color=alt.Color('cardiovascular_risk', legend=alt.Legend(title="Risk Group"),
                    scale=alt.Scale(domain=[0, 1], range=['steelblue', 'firebrick'])),
    tooltip=[alt.Tooltip('age', bin=True), 'count()']
).properties(
    title='Age Distribution by Cardiovascular Risk'
)

# Cholesterol Distribution by Cardiovascular Risk
chol_chart = alt.Chart(cardio_data).mark_bar().encode(
    x=alt.X('cholesterol', bin=True, title='Cholesterol (mg/dL)'),
    y=alt.Y('count()', title='Number of Individuals'),
    color=alt.Color('cardiovascular_risk', legend=alt.Legend(title="Risk Group"),
                    scale=alt.Scale(domain=[0, 1], range=['steelblue', 'firebrick'])),
    tooltip=[alt.Tooltip('cholesterol', bin=True), 'count()']
).properties(
    title='Cholesterol Distribution by Cardiovascular Risk'
)

# Display the charts
age_chart.display()
chol_chart.display()

# 2. Preprocessing
# Split the dataset into features (X) and the target variable (y)
X = cardio_data.drop('cardiovascular_risk', axis=1)
y = cardio_data['cardiovascular_risk']

# Split X and y into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Model Training & Evaluation (Pipelines)
# Decision Tree Pipeline
dt_pipe = make_pipeline(StandardScaler(), DecisionTreeClassifier(random_state=42))

# K-Nearest Neighbors Pipeline
knn_pipe = make_pipeline(StandardScaler(), KNeighborsClassifier())

# Random Forest Classifier (no pipeline needed for scaling)
rf_model = RandomForestClassifier(random_state=42)

# 4. Hyperparameter Tuning
# Decision Tree Grid Search
dt_param_grid = {'decisiontreeclassifier__max_depth': [3, 5, 7, 9],
                 'decisiontreeclassifier__min_samples_split': [2, 5, 10]}
dt_grid = GridSearchCV(dt_pipe, dt_param_grid, cv=5, scoring='accuracy')
dt_grid.fit(X_train, y_train)

# K-Nearest Neighbors Grid Search
knn_param_grid = {'kneighborsclassifier__n_neighbors': [3, 5, 7, 9, 11]}
knn_grid = GridSearchCV(knn_pipe, knn_param_grid, cv=5, scoring='accuracy')
knn_grid.fit(X_train, y_train)

# Random Forest Grid Search
rf_param_grid = {'n_estimators': [50, 100, 200],
                 'max_depth': [5, 10, 15]}
rf_grid = GridSearchCV(rf_model, rf_param_grid, cv=5, scoring='accuracy')
rf_grid.fit(X_train, y_train)

# 5. Results & Conclusion
print("--- Decision Tree Classifier ---")
print("Best Hyperparameters:", dt_grid.best_params_)
dt_pred = dt_grid.best_estimator_.predict(X_test)
print("\nClassification Report:\n", classification_report(y_test, dt_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, dt_pred))

print("\n--- K-Nearest Neighbors Classifier ---")
print("Best Hyperparameters:", knn_grid.best_params_)
knn_pred = knn_grid.best_estimator_.predict(X_test)
print("\nClassification Report:\n", classification_report(y_test, knn_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, knn_pred))

print("\n--- Random Forest Classifier ---")
print("Best Hyperparameters:", rf_grid.best_params_)
rf_pred = rf_grid.best_estimator_.predict(X_test)
print("\nClassification Report:\n", classification_report(y_test, rf_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, rf_pred))

# Conclusion
print("\n--- Conclusion ---")
print("Based on the classification reports, the **Random Forest Classifier** performed best on this dataset.")
print("It achieved the highest precision (0.90) and f1-score (0.87) for the 'high risk' (class 1) group, and the highest overall accuracy (0.91).")
print("This indicates that the Random Forest model is the most effective at correctly identifying patients at high risk of cardiovascular disease while maintaining high overall performance.")

ImportError: The vl-convert Vega-Lite compiler and file export feature requires
version 1.6.0 or greater of the 'vl-convert-python' package. 
This can be installed with pip using:
   pip install "vl-convert-python>=1.6.0"
or conda:
   conda install -c conda-forge "vl-convert-python>=1.6.0"

ImportError: vl-convert-python

alt.Chart(...)

ImportError: The vl-convert Vega-Lite compiler and file export feature requires
version 1.6.0 or greater of the 'vl-convert-python' package. 
This can be installed with pip using:
   pip install "vl-convert-python>=1.6.0"
or conda:
   conda install -c conda-forge "vl-convert-python>=1.6.0"

ImportError: vl-convert-python

alt.Chart(...)

--- Decision Tree Classifier ---
Best Hyperparameters: {'decisiontreeclassifier__max_depth': 5, 'decisiontreeclassifier__min_samples_split': 2}

Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.94      0.90       140
           1       0.82      0.68      0.75        60

    accuracy                           0.86       200
   macro avg       0.85      0.81      0.82       200
weighted avg       0.86      0.86      0.86       200

Confusion Matrix:
 [[131   9]
 [ 19  41]]

--- K-Nearest Neighbors Classifier ---
Best Hyperparameters: {'kneighborsclassifier__n_neighbors': 7}

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.91      0.90       140
           1       0.77      0.73      0.75        60

    accuracy                           0.85       200
   macro avg       0.83      0.82      0.82       200
weighted avg       0.85      0.85      0.85       200

Confusi